# CPU 双语字幕运行器（可迁移版）

支持 Kaggle、ModelScope DSW 和其他 Linux 云算力。模型会在首次运行时从 Hugging Face 直接下载；工作目录可通过环境变量 `ASR_WORKDIR` 覆盖。

In [57]:
from pathlib import Path
import os
import subprocess

# Kaggle 当前会话的可写工作目录
os.environ["ASR_WORKDIR"] = "/kaggle/working/asr"

# 不写死平台路径：默认使用当前工作目录下的 asr。
# 在不同云平台可先设置 os.environ['ASR_WORKDIR'] = '/持久化目录/asr'。
WORKDIR = Path(os.environ.get('ASR_WORKDIR', Path.cwd() / 'asr')).expanduser()
WORKDIR.mkdir(parents=True, exist_ok=True)
%cd {WORKDIR}
print(f'工作目录：{WORKDIR}')

def run(command):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, check=True)

run(['python', '-m', 'pip', 'install', '-U', 'pip'])
run(['python', '-m', 'pip', 'install', 'faster-whisper', 'av', 'ctranslate2', 'tokenizers', 'onnxruntime', 'numpy', 'requests', 'tqdm', 'huggingface_hub'])

llama_dir = WORKDIR / 'llama_cpp'
archive = WORKDIR / 'llama.cpp.tar.gz'
llama_dir.mkdir(exist_ok=True)
if not any(path.is_file() for path in llama_dir.rglob('llama-server')):
    run(['wget', '-c', '--show-progress', '--timeout=30', '--tries=3', 'https://github.com/ggml-org/llama.cpp/releases/download/b10516/llama-b10516-bin-ubuntu-x64.tar.gz', '-O', str(archive)])
    run(['tar', '-xzf', str(archive), '-C', str(llama_dir)])
    archive.unlink(missing_ok=True)

servers = [path for path in llama_dir.rglob('llama-server') if path.is_file()]
assert servers, '未找到 llama-server'
servers[0].chmod(0o755)
print(f'准备完成：{servers[0]}')


/kaggle/working/asr
工作目录：/kaggle/working/asr
+ python -m pip install -U pip
+ python -m pip install faster-whisper av ctranslate2 tokenizers onnxruntime numpy requests tqdm huggingface_hub
准备完成：/kaggle/working/asr/llama_cpp/llama-b10516/llama-server


## 自动下载公开模型
在 Kaggle 中请先在 Session options 开启 Internet；已存在的模型会自动跳过下载。

In [58]:
from huggingface_hub import snapshot_download, hf_hub_download

WHISPER_MODEL = WORKDIR / 'whisper' / 'medium'
TRANSLATION_MODEL = WORKDIR / 'models' / 'Hy-MT2-1.8B-GGUF' / 'Hy-MT2-1.8B-Q4_K_M.gguf'

if not (WHISPER_MODEL / 'model.bin').exists():
    snapshot_download(repo_id='Systran/faster-whisper-medium', local_dir=str(WHISPER_MODEL))
else:
    print(f'Whisper 模型已存在：{WHISPER_MODEL}')

if not TRANSLATION_MODEL.exists():
    TRANSLATION_MODEL.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(repo_id='tencent/Hy-MT2-1.8B-GGUF', filename='Hy-MT2-1.8B-Q4_K_M.gguf', local_dir=str(TRANSLATION_MODEL.parent))
else:
    print(f'翻译模型已存在：{TRANSLATION_MODEL}')


Whisper 模型已存在：/kaggle/working/asr/whisper/medium
翻译模型已存在：/kaggle/working/asr/models/Hy-MT2-1.8B-GGUF/Hy-MT2-1.8B-Q4_K_M.gguf


## 检查脚本和视频
将 `transcribe_translate_bilingual.py` 与 MP4 文件放进上述工作目录；它们是用户内容，不自动下载。

In [59]:
SCRIPT = WORKDIR / 'transcribe_translate_bilingual.py'
LLAMA_SERVER = next(path for path in (WORKDIR / 'llama_cpp').rglob('llama-server') if path.is_file())
VIDEO_DIR = Path("/kaggle/input/datasets/wangzhi2003cn163com/videos")
OUTPUT_DIR = WORKDIR / 'outputs'

for item in (WHISPER_MODEL, TRANSLATION_MODEL, SCRIPT, LLAMA_SERVER, VIDEO_DIR):
    print(('OK   ' if item.exists() else 'MISS '), item)
assert all(item.exists() for item in (WHISPER_MODEL, TRANSLATION_MODEL, SCRIPT, LLAMA_SERVER, VIDEO_DIR)), '请先放入脚本和视频。'


OK    /kaggle/working/asr/whisper/medium
OK    /kaggle/working/asr/models/Hy-MT2-1.8B-GGUF/Hy-MT2-1.8B-Q4_K_M.gguf
OK    /kaggle/working/asr/transcribe_translate_bilingual.py
OK    /kaggle/working/asr/llama_cpp/llama-b10516/llama-server
OK    /kaggle/input/datasets/wangzhi2003cn163com/videos


In [60]:
videos = sorted(VIDEO_DIR.glob('*.mp4'))
assert videos, f'未在 {VIDEO_DIR} 找到 MP4 文件'

failed = []
for index, video in enumerate(videos, 1):
    output_dir = OUTPUT_DIR / video.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    command = ['python', str(SCRIPT), '--input', str(video), '--output-dir', str(output_dir), '--whisper-model', str(WHISPER_MODEL), '--translation-model', str(TRANSLATION_MODEL), '--llama-server', str(LLAMA_SERVER), '--threads', '8']
    print(f'[{index}/{len(videos)}] 开始：{video.name}')
    try:
        subprocess.run(command, check=True)
        print(f'完成：{video.name}')
    except subprocess.CalledProcessError:
        failed.append(video.name)
        print(f'失败：{video.name}')

if failed:
    raise RuntimeError(f'失败的视频：{failed}')
print('全部视频处理完成。')


[1/1] 开始：02 .mp4
Loading Hy-MT2 Q4_K_M...
Transcribing English with Whisper...
Whisper produced 142 segments; language=en
Translated 1/142
Translated 2/142
Translated 3/142
Translated 4/142
Translated 5/142
Translated 6/142
Translated 7/142
Translated 8/142
Translated 9/142
Translated 10/142
Translated 11/142
Translated 12/142
Translated 13/142
Translated 14/142
Translated 15/142
Translated 16/142
Translated 17/142
Translated 18/142
Translated 19/142
Translated 20/142
Translated 21/142
Translated 22/142
Translated 23/142
Translated 24/142
Translated 25/142
Translated 26/142
Translated 27/142
Translated 28/142
Translated 29/142
Translated 30/142
Translated 31/142
Translated 32/142
Translated 33/142
Translated 34/142
Translated 35/142
Translated 36/142
Translated 37/142
Translated 38/142
Translated 39/142
Translated 40/142
Translated 41/142
Translated 42/142
Translated 43/142
Translated 44/142
Translated 45/142
Translated 46/142
Translated 47/142
Translated 48/142
Translated 49/142
Trans

In [61]:
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path.name, f'{path.stat().st_size / 1024:.1f} KB')


02  4.0 KB
